## agent 调用

In [ ]:
from langchain.agents import create_agent
from langchain.chat_models import init_chat_model
from openrouter.operations import ProviderResponse
from rich import print as rprint
import os
from dotenv import load_dotenv

load_dotenv(override=True)

model = init_chat_model(
    model="deepseek-chat"
)

agent = create_agent(model=model)
# rprint(agent)

response = agent.invoke({"messages": ["你好，你是谁(三句话形容)"]})
rprint(response)

# 工具调用

In [ ]:

from fastmcp.tools import tool


@tool("get_weather")
def get_weather(city_name: str, tempt: str, park: str) -> str:
    """
    天气查询工具
        Args:
            city_name : 城市名称
            tempt : 推荐景区
            park : 推荐公园
    """
    return f"{city_name}的天气为晴朗。" + f"景区是是：{tempt}" + f"公园是：{park}"


agent = create_agent(model=model,
                     name="chat_Cat",

                     system_prompt="你是一个知识助手，可以调用工具帮助用户解决问题"
                     )
response = agent.invoke(
    {
        # "messages": [
        #     {"role": "system", "content": "你是一个推荐助手"},
        #     {"role": "user",
        #      "content": "天津的天气怎么样？有什么可以推荐的景区和公园，只推荐一个只需要说出名字，不需要过多陈述"}
        # ]
        "messages": ["你是谁"
                     ]
    }
)
rprint(response)

In [ ]:

from langchain_core.messages import HumanMessage
from langchain_tavily import TavilySearch
from dotenv import load_dotenv
import os

load_dotenv(override=True)
web_search = TavilySearch(
    tavily_api_key=os.getenv("TAVILY_API_KEY"),
    max_results=1
)
agent = create_agent(model=model,
                     name="chat_Cat",
                     tools=[web_search],
                     system_prompt="你是一个知识助手，可以调用工具帮助用户解决问题,喜欢用繁体字回答"
                     )
response = agent.invoke(
    {"messages": [HumanMessage(content="奥运会是哪个国家举办的？")]}
)

rprint(response['messages'][-1].content)

## 放到工具中

In [ ]:
agent = create_agent(model=model,
                     tools=[web_search])

response = agent.invoke(
    {
        "messages": [{"role": "user", "content": "请帮我查询2024年诺贝尔物理学奖得主是谁？"}]
    }
)

rprint(response['messages'][-1].content)

# 多个工具

In [ ]:

@tool("get_news")
def get_news(topic: str, news: str, time: str) -> str:
    """
    新闻助手
        Args:
            topic : 新闻主题的名字
            news  : 新闻的内容
            time  : 新闻的时间
    """
    return f"新闻主题：{topic}，内容：{news}，时间：{time}"


@tool("get_fruits")
def get_fruits(fruits_name: str) -> str:
    """
    新闻助手
        Args:
            fruits_name : 水果
    """
    return f"水果：{fruits_name}"


agent = create_agent(model=model,
                     name="chat_Cat",
                     system_prompt="你是一个善于解决问题的助手，喜欢用繁体字说",
                     tools=[get_news, web_search, get_fruits])

response = agent.invoke(
    {"messages": ["漫步者的打游戏耳机推荐、今日国际新闻、今天适合吃什么水果？各用一句话回答"]},
    config={"recursion_limit": 10}  # 限制工具调用次数
)
rprint(response['messages'][-1].content)


# 创建结构化输出

In [38]:

from pydantic import BaseModel, Field
from langchain.agents import create_agent
from langchain.agents.structured_output import ProviderStrategy, ToolStrategy

model = init_chat_model(
    model="deepseek-chat"
)


class ContactInfo(BaseModel):
    """用户的联系方式"""
    name: str = Field(description="用户姓名")
    email: str = Field(description="用户邮箱地址")
    phone: str = Field(description="用户的手机号")


@tool
def search_tool(query: str) -> str:


    """这是一个搜索引擎。当大模型发现给定的上下文里缺少必要的联系人信息，
    需要去互联网上查询时，才会调用这个工具。
    """
    return f"搜索结果: 未找到关于 '{query}' 的更多额外信息。"

agent = create_agent(model=model,
                     tools=[search_tool],
                     response_format=ToolStrategy(ContactInfo))
response = agent.invoke({
    "messages": [HumanMessage("从这段话中抽取结构化信息：小明的邮箱地址为：shkstart@atguigu .com，手机号：12345678912")
                 ]
}
)
rprint(response["structured_response"])

ContactInfo(name='小明', email='shkstart@atguigu .com', phone='12345678912')